# Lab 12 — Spark SQL, DataFrame API, and Hive

**Tujuan:** Membandingkan SQL/DataFrame plans serta mengenal tabel dan metadata.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 12")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

In [ ]:
transactions = spark.range(0, 100_000, 1, 4)
transactions = transactions.withColumn("customer_id", F.col("id") % 1000)
transactions = transactions.withColumn("amount", (F.col("id") % 5000) + 100)
transactions.createOrReplaceTempView("transactions")

## 1. SQL expression

In [ ]:
sql_result = spark.sql("""
    SELECT customer_id, SUM(amount) AS total_amount
    FROM transactions
    GROUP BY customer_id
""")
sql_result.explain("formatted")
sql_result.show(5)

## 2. Equivalent DataFrame API

In [ ]:
df_result = transactions.groupBy("customer_id").sum("amount")
df_result.explain("formatted")
df_result.show(5)

## 3. Inspect catalog available to current Spark session

`createOrReplaceTempView` bukan Hive persisted table. Mengecek apakah Spark terhubung ke metastore; tidak mengubah konfigurasi Compose.

In [ ]:
print("Catalog implementation:", spark.conf.get("spark.sql.catalogImplementation"))
spark.sql("SHOW DATABASES").show(truncate=False)
spark.sql("SHOW TABLES").show(truncate=False)

## 4. Optional: persist as Parquet and query as SQL table

Ini membuat **temporary view dari Parquet**, bukan managed Hive table. Membuat Hive table memerlukan session dengan Hive support dan koneksi metastore yang kompatibel.

In [ ]:
path = "/home/spark/outputs/lab12/transactions"
transactions.write.mode("overwrite").parquet(path)
from_files = spark.read.parquet(path)
from_files.createOrReplaceTempView("transactions_parquet")
spark.sql("SELECT COUNT(*) AS rows FROM transactions_parquet").show()

**Tugas:** jelaskan perbedaan temp view, file Parquet, dan Hive metastore table. Hive metastore dalam Compose tidak otomatis berarti `enableHiveSupport()` telah dikonfigurasi.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.